# 🧬 Boltz-2 Structure Prediction Example

This notebook demonstrates how to predict protein and protein-ligand complex structures using **Boltz-2**.

## 📖 What is Boltz-2?

Boltz-2 is an open-source biomolecular structure prediction model developed by MIT. It predicts structures of proteins, nucleic acids (DNA/RNA), and ligands, as well as their complexes.

### Key Features:
- **Multi-modal** -- Supports proteins, DNA, RNA, and ligands
- **Complex prediction** -- Handles protein-ligand, protein-DNA/RNA, and multi-chain complexes
- **Confidence scores** -- Provides metrics including pTM, ipTM, ligand_ipTM, and pLDDT
- **MSA-optional** -- Can run with or without multiple sequence alignments
- **Fast** -- Efficient inference with flexible recycling options

## 📥 Imports

## 📦 Shared Data Types

### `StructurePredictionComplex`
A biological complex containing one or more molecular chains to predict together.

| Field | Type | Description |
|-------|------|-------------|
| `chains` | `List[Chain]` | Chains in the complex. Accepts strings, `Chain` objects, or dicts |

### `Chain`
A single molecular chain.

| Field | Type | Description |
|-------|------|-------------|
| `sequence` | `str` | Sequence (protein amino acids, DNA/RNA bases, or ligand SMILES) |
| `entity_type` | `Optional[str]` | `"protein"`, `"dna"`, `"rna"`, or `"ligand"`. Auto-inferred if `None` |

### `Structure`
A predicted 3D structure with coordinates, confidence metrics, and export methods.

## 📥 Imports

In [1]:
from pathlib import Path

from bio_programming_tools import (
    Boltz2Config,
    Boltz2Input,
    Chain,
    StructurePredictionComplex,
    run_boltz2,
)

## 🔍 Protein-Ligand Complex Prediction (MfnG Example)

Let's predict the structure of MfnG protein bound to L-tyrosine ligand.

### API Reference

**`Boltz2Input`**

| Field | Type | Description |
|-------|------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | List of complexes to predict structures for |

**`Boltz2Config`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `use_msa` | `bool` | `True` | Whether to generate and use MSAs for protein chains using ColabFold search |
| `recycling_steps` | `int` | `10` | Number of iterative refinement passes (higher=more refined but slower) |
| `sampling_steps` | `int` | `200` | Number of denoising steps in the diffusion process (higher=more refined but slower) |
| `diffusion_samples` | `int` | `25` | Number of independent structure samples per complex (only best is returned) |

**`Boltz2Output`**

| Field | Type | Description |
|-------|------|-------------|
| `structures` | `List[Structure]` | List of predicted structures, one per input complex |

In [2]:
# MfnG protein sequence
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = StructurePredictionComplex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = Boltz2Input(complexes=[complex])

# Configure Boltz-2
config = Boltz2Config(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    use_msa=True,
    recycling_steps=3,
    diffusion_samples=1,
)

# Run prediction
result = run_boltz2(inputs, config)
mfng_structure = result.structures[0]

# Print metrics
print(f"\n✅ Structure predicted!")
print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length: {len(mfng_sequence)} residues")
print(f"  Confidence score: {mfng_structure.confidence_score:.3f}")
print(f"  Average pLDDT: {mfng_structure.avg_plddt:.3f}")
print(f"  pTM score: {mfng_structure.ptm:.3f}")
print(f"  ipTM score: {mfng_structure.iptm:.3f}")
print(f"  Ligand ipTM: {mfng_structure.ligand_iptm:.3f}")

Folding structures (Boltz-2): 100%|██████████| 1/1 [00:46<00:00, 46.73s/complex]


✅ Structure predicted!
  Number of chains: 2
  Protein length: 384 residues
  Confidence score: 0.909
  Average pLDDT: 0.918
  pTM score: 0.925
  ipTM score: 0.874
  Ligand ipTM: 0.874


### 🎨 Visualize MfnG-Ligand Complex

In [3]:
mfng_structure.visualize(style='stick', color_by='chain')

## 💾 Export Results

Save predicted structures for further analysis in other tools like PyMOL, ChimeraX, or VMD.

### Supported formats:
- **PDB** -- Standard protein data bank format, widely compatible
- **mmCIF** -- Modern crystallographic information file, supports larger structures

The B-factor column contains the pLDDT scores for confidence visualization.

In [ ]:
# Create output directory
output_dir = Path("./boltz2_outputs")
output_dir.mkdir(exist_ok=True)

# Export results to pdb files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")